RNN Part 2
SimpleRNN, Long Short Term Memory (LSTM) and Gated Recurrent Unit (GRU)

In [2]:
from google.colab import files
uploaded = files.upload()

Saving quote_dataset.csv to quote_dataset.csv


In [4]:
import numpy as np
import matplotlib.pyplot as mat
import pandas as pd
import seaborn as sns

I will perform RNN in whole text dataframe this time.

In [5]:
df= pd.read_csv('quote_dataset.csv')

In [6]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [7]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Model
from keras.layers import Input, Dense, Dropout, Embedding, SimpleRNN, LSTM, GRU
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical

In [8]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [68]:
df["quote"] [0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [69]:
df.shape

(3025, 2)

Data Cleaning (Converting all the texts into lower case and removing all the punctuations)

In [9]:
quotes= quotes.str.lower()
import string
quotes = quotes.str.translate(str.maketrans('', '', string.punctuation))
quotes = quotes.str.replace(r"[“”‘’]", "", regex=True)
quotes.head()

,quote
0,the world as we have created it is a process o...
1,it is our choices harry that show what we trul...
2,there are only two ways to live your life one ...
3,the person be it gentleman or lady who has not...
4,imperfection is beauty madness is genius and i...


In [10]:
vocab_size= 10000
tokenizer= Tokenizer(num_words= vocab_size, oov_token= 'OOV')
tokenizer.fit_on_texts(quotes)

In order to guess the vocab_size, I will find the number of unique words in the dataset.

8472


[('OOV', 1),
 ('the', 2),
 ('you', 3),
 ('to', 4),
 ('and', 5),
 ('a', 6),
 ('i', 7),
 ('is', 8),
 ('of', 9),
 ('that', 10)]

In [12]:
sequences= tokenizer.texts_to_sequences(quotes)

In [70]:
for i in range(3):
    print(quotes[i])
    print(sequences[i])

the world as we have created it is a process of our thinking it cannot be changed without changing our thinking
[2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9, 71, 286, 11, 147, 13, 802, 106, 742, 71, 286]
it is our choices harry that show what we truly are far more than our abilities
[11, 8, 71, 861, 374, 10, 435, 22, 20, 466, 15, 297, 53, 55, 71, 2402]
there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle
[43, 15, 54, 202, 707, 4, 82, 16, 37, 38, 8, 30, 321, 96, 8, 6, 1027, 2, 103, 8, 30, 321, 129, 8, 6, 1027]


Create multiple sequences/ sequence from multiple words

In [13]:
X= []
y= []
for seq in sequences:
    for i in range(1, len(seq)):
        input_seq= seq[:i]
        output_seq= seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [71]:
X

[[2],
 [2, 63],
 [2, 63, 30],
 [2, 63, 30, 20],
 [2, 63, 30, 20, 17],
 [2, 63, 30, 20, 17, 934],
 [2, 63, 30, 20, 17, 934, 11],
 [2, 63, 30, 20, 17, 934, 11, 8],
 [2, 63, 30, 20, 17, 934, 11, 8, 6],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9, 71],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9, 71, 286],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9, 71, 286, 11],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9, 71, 286, 11, 147],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9, 71, 286, 11, 147, 13],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9, 71, 286, 11, 147, 13, 802],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9, 71, 286, 11, 147, 13, 802, 106],
 [2,
  63,
  30,
  20,
  17,
  934,
  11,
  8,
  6,
  1138,
  9,
  71,
  286,
  11,
  147,
  13,
  802,
  106,
  742],
 [2,
  63,
  30,
  20,
  17,
  934,
  11,
  8,
  6,
  1138,
  9,
  71,
  286,
  11,
  147,
  13,
  802,
  106,
  742,
  71],
 [11],
 [11, 8],
 [11

In [72]:
y

array([ 63,  30,  20, ...,   4, 173, 103])

In [73]:
len(X), len(y)

(84823, 84823)

In [14]:
max_length= max(len(x) for x in X)
max_length

745

In [15]:
X_pad= pad_sequences(X, maxlen= max_length, padding= "pre")
y= np.array(y)

In [74]:
X_pad.shape, y.shape

((84823, 745), (84823,))

One-Hot Encoding for y

In [16]:
y_encoded= to_categorical(y, num_classes= vocab_size)

In [17]:
X_pad.shape, y_encoded.shape

((84823, 745), (84823, 10000))

In [18]:
embed_dim= 32
rnn_units= 64

Model Creation and Training for SimpleRNN.

In [19]:
input_layer= Input(shape= (max_length,), dtype= "int32", name= "Input_Layer")
embedding_layer= Embedding(input_dim= vocab_size, output_dim= embed_dim, mask_zero= True, name= "Embedding_Layer")(input_layer)

In [20]:
rnn= SimpleRNN(units= rnn_units, return_sequences= False, return_state= False, name= "SimpleRNN_Layer")
embedding_layer_last= rnn(embedding_layer)
output_layer= Dense(units= vocab_size, activation= "softmax", name= "Output_Layer")(embedding_layer_last)

In [21]:
model= Model(inputs= input_layer, outputs= output_layer)

In [22]:
model.compile(optimizer= "adam", loss= "categorical_crossentropy", metrics= ["accuracy"])

In [27]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input_Layer         │ (None, 745)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Embedding_Layer     │ (None, 745, 32)   │    320,000 │ Input_Layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 745)       │          0 │ Input_Layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ SimpleRNN_Layer     │ (None, 64)        │      6,208 │ Embedding_Layer[… │
│ (SimpleRNN)         │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Output_Layer        │ (None, 10000)     │    650,000 │ SimpleRNN_Layer[… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 976,208 (3.72 MB)

 Trainable params: 976,208 (3.72 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
simple_rnn_model= model.fit(X_pad, y_encoded, epochs= 100, batch_size= 256, validation_split= 0.2, verbose= 1)

Epoch 1/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 34s 105ms/step - accuracy: 0.0380 - loss: 7.0047 - val_accuracy: 0.0373 - val_loss: 6.7388
Epoch 2/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 21s 79ms/step - accuracy: 0.0391 - loss: 6.5432 - val_accuracy: 0.0375 - val_loss: 6.7637
Epoch 3/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 22s 84ms/step - accuracy: 0.0457 - loss: 6.4253 - val_accuracy: 0.0545 - val_loss: 6.6801
Epoch 4/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 40s 80ms/step - accuracy: 0.0598 - loss: 6.2564 - val_accuracy: 0.0668 - val_loss: 6.6041
Epoch 5/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 44s 88ms/step - accuracy: 0.0753 - loss: 6.1021 - val_accuracy: 0.0796 - val_loss: 6.5303
Epoch 6/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 22s 84ms/step - accuracy: 0.0880 - loss: 5.9542 - val_accuracy: 0.0845 - val_loss: 6.4770
Epoch 7/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 21s 79ms/step - accuracy: 0.1004 - loss: 5.8193 - val_accuracy: 0.0939 - val_loss: 6.4301
Epoch 8/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 21s 79ms/step - accuracy: 0.1121 - loss: 

For deep learning, I should save HDF5 file instead of pickel file.

In [30]:
model.save('simple_rnn_model.h5')

Model Creation and Training for LSTM.

In [23]:
input_layer2= Input(shape= (max_length,), dtype= "int32", name= "Input_Layer2")
embedding_layer2= Embedding(input_dim= vocab_size, output_dim= embed_dim, mask_zero= False, name= "Embedding_Layer2")(input_layer2)

In [24]:
rnn2= LSTM(units= rnn_units, return_sequences= False, return_state= False, name= "LSTM_Layer")
embedding_layer_last2= rnn2(embedding_layer2)
output_layer2= Dense(units= vocab_size, activation= "softmax", name= "Output_Layer2")(embedding_layer_last2)

In [25]:
model2= Model(inputs= input_layer2, outputs= output_layer2)

In [26]:
model2.compile(optimizer= "adam", loss= "categorical_crossentropy", metrics= ["accuracy"])

In [27]:
model2.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input_Layer2 (InputLayer)       │ (None, 745)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Embedding_Layer2 (Embedding)    │ (None, 745, 32)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ LSTM_Layer (LSTM)               │ (None, 64)             │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer2 (Dense)           │ (None, 10000)          │       650,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 994,832 (3.79 MB)

 Trainable params: 994,832 (3.79 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
lstm_model= model2.fit(X_pad, y_encoded, epochs= 100, batch_size= 256, validation_split= 0.2, verbose= 1)

Epoch 1/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 22s 60ms/step - accuracy: 0.0387 - loss: 6.9390 - val_accuracy: 0.0373 - val_loss: 6.6144
Epoch 2/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 15s 57ms/step - accuracy: 0.0391 - loss: 6.4188 - val_accuracy: 0.0373 - val_loss: 6.6406
Epoch 3/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 21s 79ms/step - accuracy: 0.0441 - loss: 6.3369 - val_accuracy: 0.0463 - val_loss: 6.6402
Epoch 4/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 15s 56ms/step - accuracy: 0.0487 - loss: 6.2548 - val_accuracy: 0.0523 - val_loss: 6.6135
Epoch 5/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 15s 56ms/step - accuracy: 0.0550 - loss: 6.1740 - val_accuracy: 0.0582 - val_loss: 6.5670
Epoch 6/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 15s 56ms/step - accuracy: 0.0654 - loss: 6.0921 - val_accuracy: 0.0617 - val_loss: 6.5388
Epoch 7/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 15s 56ms/step - accuracy: 0.0732 - loss: 6.0150 - val_accuracy: 0.0714 - val_loss: 6.5090
Epoch 8/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 20s 55ms/step - accuracy: 0.0815 - loss: 5

In [29]:
model2.save('lstm_model.h5')

Model Creation and Training for GRU.

In [30]:
input_layer3= Input(shape= (max_length,), dtype= "int32", name= "Input_Layer3")
embedding_layer3= Embedding(input_dim= vocab_size, output_dim= embed_dim, mask_zero= False, name= "Embedding_Layer3")(input_layer3)

In [31]:
rnn3= GRU(units= rnn_units, return_sequences= False, return_state= False, name= "GRU_Layer")
embedding_layer_last3= rnn3(embedding_layer3)
output_layer3= Dense(units= vocab_size, activation= "softmax", name= "Output_Layer3")(embedding_layer_last3)

In [32]:
model3= Model(inputs= input_layer3, outputs= output_layer3)

In [33]:
model3.compile(optimizer= "adam", loss= "categorical_crossentropy", metrics= ["accuracy"])

In [34]:
model3.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input_Layer3 (InputLayer)       │ (None, 745)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Embedding_Layer3 (Embedding)    │ (None, 745, 32)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ GRU_Layer (GRU)                 │ (None, 64)             │        18,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer3 (Dense)           │ (None, 10000)          │       650,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 988,816 (3.77 MB)

 Trainable params: 988,816 (3.77 MB)

 Non-trainable params: 0 (0.00 B)

In [35]:
gru_model= model3.fit(X_pad, y_encoded, epochs= 100, batch_size= 256, validation_split= 0.2, verbose= 1)

Epoch 1/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 18s 62ms/step - accuracy: 0.0382 - loss: 6.9896 - val_accuracy: 0.0373 - val_loss: 6.6979
Epoch 2/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 15s 56ms/step - accuracy: 0.0397 - loss: 6.4746 - val_accuracy: 0.0472 - val_loss: 6.6297
Epoch 3/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 14s 53ms/step - accuracy: 0.0561 - loss: 6.2576 - val_accuracy: 0.0670 - val_loss: 6.5100
Epoch 4/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 15s 57ms/step - accuracy: 0.0742 - loss: 6.0738 - val_accuracy: 0.0822 - val_loss: 6.4454
Epoch 5/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 14s 52ms/step - accuracy: 0.0892 - loss: 5.9301 - val_accuracy: 0.0883 - val_loss: 6.4181
Epoch 6/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 15s 54ms/step - accuracy: 0.0967 - loss: 5.8072 - val_accuracy: 0.0899 - val_loss: 6.3974
Epoch 7/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 14s 54ms/step - accuracy: 0.1050 - loss: 5.6973 - val_accuracy: 0.0933 - val_loss: 6.3866
Epoch 8/100
266/266 ━━━━━━━━━━━━━━━━━━━━ 14s 53ms/step - accuracy: 0.1113 - loss: 5

In [36]:
model3.save('gru_model.h5')

Apply the trained model from H5 file

In [37]:
index_to_word= {}
for word, index in word_index.items():
    index_to_word[index]= word

In [60]:
def predictor(model, tokenizer, text, max_length):
    text= text.lower()
    seq= tokenizer.texts_to_sequences([text]) [0]
    seq= pad_sequences([seq], maxlen= max_length, padding= "pre")
    pred= model.predict(seq, verbose=1)
    pred_index= np.argmax(pred, axis= -1) [0]
    return index_to_word.get(pred_index, "")

In [61]:
seed_text= "you will"
next_word= predictor(model, tokenizer, seed_text, max_length)
print(next_word)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
driven


In [62]:
seed_text= "you will"
next_word= predictor(model2, tokenizer, seed_text, max_length)
print(next_word)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
be


In [63]:
seed_text= "you will"
next_word= predictor(model3, tokenizer, seed_text, max_length)
print(next_word)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
not


Predict a whole sentence from a word

In [64]:
def generate_text(model, tokenizer, seed_text, max_length, num_words):
    for _ in range(num_words):
        next_word= predictor(model, tokenizer, seed_text, max_length)
        if next_word == "":
          break
        seed_text += " " + next_word
    return seed_text

In [65]:
seed_text= "the meaning of life"
generated_text= generate_text(model, tokenizer, seed_text, max_length, 10)
print(generated_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
the meaning of life anytime oncerequited faced


In [66]:
seed_text= "the meaning of life"
generated_text= generate_text(model2, tokenizer, seed_text, max_length, 10)
print(generated_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
the meaning of life is to be a good book is a good piece


In [67]:
seed_text= "the meaning of life"
generated_text= generate_text(model3, tokenizer, seed_text, max_length, 10)
print(generated_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
the meaning of life is the cosmic perspective that you can afford to be


Now, I will use a model in an User Interface (UI) i.e. Model Deployment.

For this I will use pickle file.

In [75]:
import pickle

In [76]:
with open('tokenizer.pickle', 'wb') as f:
    pickle.dump(tokenizer, f)

In [77]:
with open("max_length.pickle", "wb") as f:
    pickle.dump(max_length, f)

Now, I will create an app for next word prediction, from .py file in VS Code.

Since, the accuracy from GRU is higher, I will use it's .h5 file to create UI.